# AI Coach — POC: Import Garmin Activities via Garmin Connect API

Credentials werden sicher per Eingabe abgefragt. Die Aktivitäten werden direkt von
Garmin Connect heruntergeladen und als `.fit` Dateien in `data/activities/` gespeichert.

In [1]:
import sys
sys.path.insert(0, "..")

from pathlib import Path
import pandas as pd

from src.garmin_client import GarminClient
from src.fit_parser import parse_fit_file
from src.data_utils import expand_timestamp, reorder_columns, missing_data_report, drop_high_missing
from src.wellness import fetch_wellness_data

In [2]:
# --- Config ---
ACTIVITIES_DIR = Path("../data/activities")
TOKEN_STORE    = Path("../data/.garmin_tokens")
NUM_ACTIVITIES = 100

# --- Login (mit Token-Cache) ---
# Beim ersten Aufruf: E-Mail/Passwort eingeben → Token wird gespeichert.
# Ab dem zweiten Aufruf: Token wird automatisch geladen, keine Eingabe nötig.
gc = GarminClient(token_store=TOKEN_STORE)
gc.login()

# --- FIT-Dateien herunterladen ---
fit_files  = gc.download_activities(ACTIVITIES_DIR, num=NUM_ACTIVITIES)
activities = gc.activities

Login erfolgreich.
  Bereits vorhanden: 22892213122.fit
  Bereits vorhanden: 22869772148.fit
  Bereits vorhanden: 22845110543.fit
  Bereits vorhanden: 22818961631.fit
  Bereits vorhanden: 22798371724.fit
  Bereits vorhanden: 22777198973.fit
  Bereits vorhanden: 22777198168.fit
  Bereits vorhanden: 22746149597.fit
  Bereits vorhanden: 22736420829.fit
  Bereits vorhanden: 22715678237.fit
  Bereits vorhanden: 22691212409.fit
  Bereits vorhanden: 22675360493.fit
  Bereits vorhanden: 22651476092.fit
  Bereits vorhanden: 22631636721.fit
  Bereits vorhanden: 22607649768.fit
  Bereits vorhanden: 22572922875.fit
  Bereits vorhanden: 22563039853.fit
  Bereits vorhanden: 22550078368.fit
  Bereits vorhanden: 22524864188.fit
  Bereits vorhanden: 22524862706.fit
  Bereits vorhanden: 22478650961.fit
  Bereits vorhanden: 22455357310.fit
  Bereits vorhanden: 22418573707.fit
  Bereits vorhanden: 22395566165.fit
  Bereits vorhanden: 22361089765.fit
  Bereits vorhanden: 22330086513.fit

Gesamt: 26 .fit Da

In [3]:
fit_files[0]

WindowsPath('../data/activities/22330086513.fit')

In [4]:
# Parse the first available file as a quick test
if fit_files:
    df = parse_fit_file(fit_files[0])
    print(f"Parsed: {fit_files[0].name}  →  {len(df)} records, {len(df.columns)} fields")
    print("\nColumns:", list(df.columns))
    df.head()
else:
    print("No .fit files found. Copy files from your Garmin device to:", ACTIVITIES_DIR.resolve())

Parsed: 22330086513.fit  →  146 records, 9 fields

Columns: ['sport', 'sub_sport', 'distance', 'enhanced_altitude', 'heart_rate', 'position_lat', 'position_long', 'timestamp', 'enhanced_speed']


In [5]:
# Parse ALL .fit files into a single combined DataFrame
# activity_meta liefert den offiziellen Garmin-Aktivitätstyp aus der API
activity_meta = {
    str(act["activityId"]): act.get("activityType", {}).get("typeKey", "unknown")
    for act in activities
}

all_dfs = []
for f in fit_files:
    df_tmp = parse_fit_file(f)
    act_id_str = f.stem  # Dateiname ohne .fit = activityId
    fallback = df_tmp["sport"].iloc[0] if len(df_tmp) else "unknown"
    df_tmp = df_tmp.assign(
        activity_type=activity_meta.get(act_id_str, fallback),
        source_file=f.name,
    )
    cols = ["activity_type", "source_file"] + [c for c in df_tmp.columns if c not in ("activity_type", "source_file")]
    all_dfs.append(df_tmp[cols])

if all_dfs:
    df_all = pd.concat(all_dfs, ignore_index=True)
    print(f"Total records: {len(df_all)}")
    print(f"\nAktivitäten:")
    print(df_all.groupby("source_file")["activity_type"].first().to_string())
    print(f"\nSpalten ({len(df_all.columns)}): {list(df_all.columns)}")
    df_all.head()
else:
    print("No .fit files to process.")

Total records: 51786

Aktivitäten:
source_file
22330086513.fit              cycling
22361089765.fit              running
22395566165.fit         lap_swimming
22418573707.fit              running
22455357310.fit              cycling
22478650961.fit    treadmill_running
22524862706.fit       indoor_cycling
22524864188.fit              running
22550078368.fit               hiking
22563039853.fit              running
22572922875.fit              cycling
22607649768.fit              running
22631636721.fit       indoor_cycling
22651476092.fit              cycling
22675360493.fit       indoor_cycling
22691212409.fit              cycling
22715678237.fit              running
22736420829.fit              cycling
22746149597.fit              running
22777198168.fit       indoor_cycling
22777198973.fit              running
22798371724.fit              running
22818961631.fit              cycling
22845110543.fit              running
22869772148.fit              running
22892213122.fit       indoor

In [6]:
df_all.head()

,activity_type,source_file,sport,sub_sport,distance,enhanced_altitude,heart_rate,position_lat,position_long,timestamp,...,fractional_cadence,power,stance_time,stance_time_balance,stance_time_percent,step_length,vertical_oscillation,vertical_ratio,accumulated_power,temperature
0,cycling,22330086513.fit,cycling,generic,0.0,104.6,98,NaN,NaN,2026-03-28 15:50:55,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cycling,22330086513.fit,cycling,generic,0.0,104.6,97,NaN,NaN,2026-03-28 15:50:56,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cycling,22330086513.fit,cycling,generic,0.0,104.6,94,NaN,NaN,2026-03-28 15:51:02,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cycling,22330086513.fit,cycling,generic,0.0,104.6,97,NaN,NaN,2026-03-28 15:51:14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cycling,22330086513.fit,cycling,generic,0.0,104.6,100,NaN,NaN,2026-03-28 15:51:27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Process Training Data

In [7]:
df_all = expand_timestamp(df_all)
df_all = reorder_columns(df_all, ["source_file", "timestamp", "jahr", "monat", "tag", "uhrzeit"])

In [8]:
df_all.head()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,sub_sport,distance,...,fractional_cadence,power,stance_time,stance_time_balance,stance_time_percent,step_length,vertical_oscillation,vertical_ratio,accumulated_power,temperature
0,22330086513.fit,2026-03-28 15:50:55,2026,3,28,15:50:55,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22330086513.fit,2026-03-28 15:50:56,2026,3,28,15:50:56,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22330086513.fit,2026-03-28 15:51:02,2026,3,28,15:51:02,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22330086513.fit,2026-03-28 15:51:14,2026,3,28,15:51:14,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22330086513.fit,2026-03-28 15:51:27,2026,3,28,15:51:27,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Analyze one specific training event

In [9]:
df_all.groupby("sub_sport").count()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,distance,enhanced_altitude,...,fractional_cadence,power,stance_time,stance_time_balance,stance_time_percent,step_length,vertical_oscillation,vertical_ratio,accumulated_power,temperature
sub_sport,,,,,,,,,,,,,,,,,,,,,
generic,40050,40050,40050,40050,40050,40050,40050,40050,40050,40050,...,34729,33548,14403,0,0,19372,27588,19372,33399,0
indoor_cycling,5228,5228,5228,5228,5228,5228,5228,5228,5228,0,...,0,0,0,0,0,0,0,0,0,0
lap_swimming,1058,1058,1058,1058,1058,1058,1058,1058,0,0,...,0,0,0,0,0,0,0,0,0,0
treadmill,5450,5450,5450,5450,5450,5450,5450,5450,5450,0,...,5450,5449,84,0,0,4102,4728,4102,5446,0


In [10]:
missing_data_report(df_all)

,fehlend,anteil_%
temperature,51786,100.00
stance_time_percent,51786,100.00
stance_time_balance,51786,100.00
stance_time,37299,72.03
step_length,28312,54.67
vertical_ratio,28312,54.67
vertical_oscillation,19470,37.60
accumulated_power,12941,24.99
power,12789,24.70
position_long,11825,22.83


In [11]:
df_all = drop_high_missing(df_all, threshold=90.0)

Gelöschte Spalten (3): ['stance_time_balance', 'stance_time_percent', 'temperature']


In [12]:
df_all.head()

,source_file,timestamp,jahr,monat,tag,uhrzeit,activity_type,sport,sub_sport,distance,...,position_long,enhanced_speed,cadence,fractional_cadence,power,stance_time,step_length,vertical_oscillation,vertical_ratio,accumulated_power
0,22330086513.fit,2026-03-28 15:50:55,2026,3,28,15:50:55,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22330086513.fit,2026-03-28 15:50:56,2026,3,28,15:50:56,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22330086513.fit,2026-03-28 15:51:02,2026,3,28,15:51:02,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,22330086513.fit,2026-03-28 15:51:14,2026,3,28,15:51:14,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,22330086513.fit,2026-03-28 15:51:27,2026,3,28,15:51:27,cycling,cycling,generic,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Wellness-Daten: Resting HR & nächtliche HRV

Täglich aus der Garmin Connect API abgerufene Erholungsmetriken:

| Metrik | Quelle | Bedeutung |
|---|---|---|
| `resting_hr_bpm` | Daily Stats | Ruhepuls – Trend zeigt Übertraining / Erholung |
| `hrv_last_night` | HRV-Status | RMSSD der letzten Nacht – bester Einzelindikator für Gesamterholung |
| `hrv_weekly_avg` | HRV-Status | 7-Tage-Mittel (stabile Baseline) |
| `hrv_status` | HRV-Status | Garmin-Bewertung: BALANCED / UNBALANCED / LOW / POOR |

In [13]:
# Datum-Range aus df_all ableiten
date_start_str = df_all["timestamp"].min().strftime("%Y-%m-%d")
date_end_str   = df_all["timestamp"].max().strftime("%Y-%m-%d")

print(f"Lade Wellness-Daten von {date_start_str} bis {date_end_str} ...")
df_wellness = fetch_wellness_data(gc.raw, date_start_str, date_end_str)

print(
    f"{len(df_wellness)} Tage | "
    f"Resting HR: {df_wellness['resting_hr_bpm'].notna().sum()} Werte | "
    f"HRV lastNightAvg: {df_wellness['hrv_last_night_avg'].notna().sum()} Werte | "
    f"HRV weeklyAvg: {df_wellness['hrv_weekly_avg'].notna().sum()} Werte"
)
display(df_wellness)

Lade Wellness-Daten von 2026-03-28 bis 2026-05-15 ...
49 Tage | Resting HR: 49 Werte | HRV lastNightAvg: 49 Werte | HRV weeklyAvg: 46 Werte


,date,resting_hr_bpm,max_hr_bpm,hrv_last_night_avg,hrv_last_night_5min_high,hrv_weekly_avg,hrv_status
0,2026-03-28,59,121,31,56,NaN,NONE
1,2026-03-29,55,107,39,73,NaN,NONE
2,2026-03-30,57,118,39,66,NaN,NONE
3,2026-03-31,54,158,36,53,36.0,NONE
4,2026-04-01,56,117,39,69,37.0,NONE
5,2026-04-02,57,139,38,62,37.0,NONE
6,2026-04-03,56,141,35,54,37.0,NONE
7,2026-04-04,58,138,35,48,37.0,NONE
8,2026-04-05,54,148,38,53,37.0,NONE
9,2026-04-06,55,112,39,66,37.0,NONE
